# Análise Exploratória - Preços de Combustíveis 2026

Este notebook usa exclusivamente as tabelas produzidas pelo pipeline do projeto a partir dos dados públicos da ANP.

**Objetivos:** entender cobertura, evolução semanal, diferenças geográficas e a relação etanol/gasolina sem digitar métricas manualmente.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA = ROOT / 'data' / 'processed' / 'precos_semanais_2026.csv'
ANALYTICS = ROOT / 'data' / 'processed' / 'analytics'

if not DATA.exists():
    raise FileNotFoundError('Execute primeiro: python -m src.pipeline')

df = pd.read_csv(DATA, low_memory=False)
df['data_inicial'] = pd.to_datetime(df['data_inicial'], errors='coerce')
df.head()

## 1. Cobertura dos dados

In [ ]:
resumo = pd.DataFrame({
    'indicador': ['linhas', 'produtos', 'semanas', 'níveis geográficos'],
    'valor': [
        len(df),
        df['produto'].nunique(),
        df['data_inicial'].nunique(),
        df['nivel_geografico'].nunique(),
    ],
})
resumo

In [ ]:
pd.crosstab(df['nivel_geografico'], df['produto'])

## 2. Evolução semanal no nível Brasil

In [ ]:
brasil = df[df['nivel_geografico'].str.lower().eq('brasil')].copy()
brasil = brasil.dropna(subset=['data_inicial', 'preco_medio_revenda'])
brasil = brasil.sort_values('data_inicial')

for unidade, por_unidade in brasil.groupby('unidade_medida', dropna=False):
    unidade_label = (
        'não informada'
        if pd.isna(unidade) or not str(unidade).strip()
        else str(unidade).strip()
    )

    fig, ax = plt.subplots(figsize=(12, 6))
    for produto, grupo in por_unidade.groupby('produto'):
        ax.plot(
            grupo['data_inicial'],
            grupo['preco_medio_revenda'],
            label=produto,
        )

    ax.set_title(
        f'Evolução semanal do preço médio de revenda - Brasil - {unidade_label}'
    )
    ax.set_xlabel('Semana')
    ax.set_ylabel(f'Preço médio de revenda ({unidade_label})')
    ax.legend()
    fig.autofmt_xdate()
    plt.show()

## 3. Ranking das UFs na última semana

In [ ]:
ranking_uf_path = ANALYTICS / 'ranking_ufs_ultima_semana.csv'
ranking_uf = pd.read_csv(ranking_uf_path)

produto_normalizado = ranking_uf['produto'].astype('string').str.strip().str.upper()
mascara_gasolina_comum = (
    produto_normalizado.eq('GASOLINA')
    | (
        produto_normalizado.str.contains('GASOLINA', na=False)
        & produto_normalizado.str.contains('COMUM', na=False)
        & ~produto_normalizado.str.contains('ADITIV', na=False)
    )
)
gasolina_comum = ranking_uf.loc[mascara_gasolina_comum].copy()

series = (
    gasolina_comum[['produto', 'unidade_medida']]
    .drop_duplicates()
    .reset_index(drop=True)
)
if len(series) != 1:
    raise ValueError(
        'O ranking não possui uma única série de gasolina comum por produto e unidade.'
    )

produto_exemplo = series.loc[0, 'produto']
unidade_exemplo = series.loc[0, 'unidade_medida']

top = (
    gasolina_comum[
        gasolina_comum['produto'].eq(produto_exemplo)
        & gasolina_comum['unidade_medida'].eq(unidade_exemplo)
    ]
    .sort_values('preco_medio_revenda', ascending=False)
    .head(10)
    .sort_values('preco_medio_revenda')
)

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(top['uf'], top['preco_medio_revenda'])
ax.set_title(
    f'10 maiores preços médios por UF - {produto_exemplo} - {unidade_exemplo}'
)
ax.set_xlabel(f'Preço médio de revenda ({unidade_exemplo})')
ax.set_ylabel('UF')
plt.show()

## 4. Relação etanol/gasolina por município

In [ ]:
ratio_path = ANALYTICS / 'etanol_gasolina_ultima_semana.csv'
ratio = pd.read_csv(ratio_path)

ratio['relacao_etanol_gasolina_pct'].describe()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(ratio['preco_gasolina_comum'], ratio['preco_etanol'])
ax.set_title('Preço do etanol × gasolina comum - última semana')
ax.set_xlabel('Preço médio da gasolina comum')
ax.set_ylabel('Preço médio do etanol')
plt.show()

## 5. Cuidados de interpretação

- Os níveis Brasil, região, estado e município têm grãos diferentes e não devem ser misturados sem filtro.
- A média mensal produzida pelo projeto é derivada das observações semanais; não substitui a série mensal oficial da ANP.
- Valores extremos são investigados, não removidos automaticamente.
- A razão etanol/gasolina é apresentada como indicador observado. Qualquer limiar econômico deve ser documentado separadamente.